In [1]:
from scapy.all import sniff

# Capture 10 packets
print("Capturing 10 packets... do some browsing now!")
packets = sniff(count=10)

print("\nCaptured packets:")
packets.summary()

Capturing 10 packets... do some browsing now!

Captured packets:
Ether / IP / TCP 3.229.10.222:https > 192.168.120.41:59254 A
Ether / IP / TCP 3.229.10.222:https > 192.168.120.41:59254 A
Ether / IP / TCP 3.229.10.222:https > 192.168.120.41:59254 PA / Raw
Ether / IP / TCP 192.168.120.41:59254 > 3.229.10.222:https PA / Raw
Ether / IP / TCP 32.194.22.47:https > 192.168.120.41:51519 PA / Raw
Ether / IP / TCP 192.168.120.41:51519 > 32.194.22.47:https PA / Raw
Ether / IP / TCP 3.229.10.222:https > 192.168.120.41:59254 A
Ether / IP / TCP 192.168.120.41:59254 > 3.229.10.222:https PA / Raw
Ether / IP / TCP 32.194.22.47:https > 192.168.120.41:51519 A
Ether / IP / TCP 32.194.22.47:https > 192.168.120.41:51519 PA / Raw


In [2]:
# Show full details of the first packet
packets[0].show()

###[ Ethernet ]###
  dst       = c4:4a:4e:4d:81:ff
  src       = 12:64:81:f6:f1:83
  type      = IPv4
###[ IP ]###
     version   = 4
     ihl       = 5
     tos       = 0x0
     len       = 40
     id        = 13521
     flags     = DF
     frag      = 0
     ttl       = 244
     proto     = tcp
     chksum    = 0xa6a
     src       = 3.229.10.222
     dst       = 192.168.120.41
     \options   \
###[ TCP ]###
        sport     = https
        dport     = 59254
        seq       = 2666998084
        ack       = 1797817471
        dataofs   = 5
        reserved  = 0
        flags     = A
        window    = 180
        chksum    = 0xc676
        urgptr    = 0
        options   = []



In [3]:
packet = packets[0]

print("Source IP:", packet['IP'].src)
print("Destination IP:", packet['IP'].dst)
print("Source Port:", packet['TCP'].sport)
print("Destination Port:", packet['TCP'].dport)
print("TCP Flags:", packet['TCP'].flags)
print("Packet Length:", len(packet))
print("TTL:", packet['IP'].ttl)

Source IP: 3.229.10.222
Destination IP: 192.168.120.41
Source Port: 443
Destination Port: 59254
TCP Flags: A
Packet Length: 54
TTL: 244


In [4]:
import pandas as pd

print("Capturing 20 packets...")
packets = sniff(count=20)

# Extract info from each packet into a list
packet_data = []

for pkt in packets:
    if pkt.haslayer('IP') and pkt.haslayer('TCP'):
        packet_data.append({
            'Source_IP': pkt['IP'].src,
            'Destination_IP': pkt['IP'].dst,
            'Source_Port': pkt['TCP'].sport,
            'Destination_Port': pkt['TCP'].dport,
            'Flags': str(pkt['TCP'].flags),
            'Length': len(pkt),
            'TTL': pkt['IP'].ttl
        })

df_live = pd.DataFrame(packet_data)
df_live

Capturing 20 packets...


,Source_IP,Destination_IP,Source_Port,Destination_Port,Flags,Length,TTL
0,34.232.56.200,192.168.120.41,443,51520,PA,108,244
1,192.168.120.41,34.232.56.200,51520,443,PA,1563,128
2,34.232.56.200,192.168.120.41,443,51520,A,54,244
3,34.232.56.200,192.168.120.41,443,51520,PA,225,244
4,192.168.120.41,34.232.56.200,51520,443,A,54,128
5,192.168.120.41,32.194.22.47,51519,443,PA,367,128
6,32.194.22.47,192.168.120.41,443,51519,PA,108,244
7,192.168.120.41,32.194.22.47,51519,443,PA,1364,128
8,32.194.22.47,192.168.120.41,443,51519,PA,512,244
9,192.168.120.41,32.194.22.47,51519,443,A,54,128


In [5]:
# Create a flow identifier (combination of source+dest IP and port)
df_live['Flow_ID'] = df_live['Source_IP'] + ':' + df_live['Source_Port'].astype(str) + ' -> ' + df_live['Destination_IP'] + ':' + df_live['Destination_Port'].astype(str)

print("Number of unique flows:", df_live['Flow_ID'].nunique())
print("\nPackets per flow:")
print(df_live['Flow_ID'].value_counts())

Number of unique flows: 6

Packets per flow:
Flow_ID
34.232.56.200:443 -> 192.168.120.41:51520    3
192.168.120.41:51519 -> 32.194.22.47:443     3
192.168.120.41:51520 -> 34.232.56.200:443    2
32.194.22.47:443 -> 192.168.120.41:51519     2
192.168.120.41:50474 -> 20.210.53.246:443    1
20.210.53.246:443 -> 192.168.120.41:50474    1
Name: count, dtype: int64


In [6]:
from scapy.all import sniff
import pandas as pd
import time
from collections import defaultdict

def capture_and_extract_features(duration=10):
    print(f"Capturing traffic for {duration} seconds...")
    
    packets = sniff(timeout=duration)
    
    flows = defaultdict(list)
    
    for pkt in packets:
        if pkt.haslayer('IP') and pkt.haslayer('TCP'):
            src = pkt['IP'].src
            dst = pkt['IP'].dst
            sport = pkt['TCP'].sport
            dport = pkt['TCP'].dport
            
            # Create a flow key (same conversation regardless of direction)
            flow_key = tuple(sorted([f"{src}:{sport}", f"{dst}:{dport}"]))
            
            flows[flow_key].append({
                'time': pkt.time,
                'length': len(pkt),
                'flags': str(pkt['TCP'].flags),
                'src': src,
                'dst': dst,
                'sport': sport,
                'dport': dport
            })
    
    print(f"Captured {len(packets)} packets across {len(flows)} flows")
    return flows

flows = capture_and_extract_features(duration=10)

Capturing traffic for 10 seconds...
Captured 343 packets across 10 flows


In [7]:
def calculate_flow_features(flows):
    flow_features = []
    
    for flow_key, packets in flows.items():
        if len(packets) < 2:
            continue  # skip flows with too few packets
        
        # Sort by time
        packets_sorted = sorted(packets, key=lambda x: x['time'])
        
        # Flow Duration (in microseconds, like CICIDS)
        duration = (packets_sorted[-1]['time'] - packets_sorted[0]['time']) * 1_000_000
        
        # Determine forward direction (first packet's source = forward)
        forward_src = packets_sorted[0]['src']
        
        fwd_packets = [p for p in packets_sorted if p['src'] == forward_src]
        bwd_packets = [p for p in packets_sorted if p['src'] != forward_src]
        
        fwd_lengths = [p['length'] for p in fwd_packets]
        bwd_lengths = [p['length'] for p in bwd_packets]
        
        # Count SYN flags
        syn_count = sum(1 for p in packets_sorted if 'S' in p['flags'])
        
        # Destination port (most important feature!)
        dest_port = packets_sorted[0]['dport']
        
        features = {
            'Destination_Port': dest_port,
            'Flow_Duration': duration if duration > 0 else 1,
            'Total_Fwd_Packets': len(fwd_packets),
            'Total_Bwd_Packets': len(bwd_packets),
            'Fwd_Packet_Length_Max': max(fwd_lengths) if fwd_lengths else 0,
            'Fwd_Packet_Length_Mean': sum(fwd_lengths)/len(fwd_lengths) if fwd_lengths else 0,
            'Bwd_Packet_Length_Mean': sum(bwd_lengths)/len(bwd_lengths) if bwd_lengths else 0,
            'Flow_Packets_per_s': len(packets_sorted) / (duration/1_000_000) if duration > 0 else 0,
            'SYN_Flag_Count': syn_count,
            'Total_Length_Fwd': sum(fwd_lengths),
        }
        
        flow_features.append(features)
    
    return pd.DataFrame(flow_features)

df_features = calculate_flow_features(flows)
df_features

,Destination_Port,Flow_Duration,Total_Fwd_Packets,Total_Bwd_Packets,Fwd_Packet_Length_Max,Fwd_Packet_Length_Mean,Bwd_Packet_Length_Mean,Flow_Packets_per_s,SYN_Flag_Count,Total_Length_Fwd
0,62767,9.608269e+01,1,1,54,54.000000,54.000000,20815.404467,0,54
1,54008,1.767924e+06,37,8,1414,509.000000,165.750000,25.453582,0,18833
2,443,5.011194e+06,9,9,1540,623.111111,129.000000,3.591958,0,5608
3,443,5.686310e+06,12,11,3931,915.916667,240.181818,4.044802,0,10991
4,443,3.320600e+06,14,31,1517,295.357143,425.096774,13.551768,2,4135
5,443,3.311456e+06,9,12,4317,1093.666667,104.750000,6.341621,2,9843
6,443,1.810015e+06,16,14,1927,488.375000,711.642857,16.574449,2,7814
7,443,5.052311e+05,7,7,571,195.857143,942.000000,27.710089,2,1371
8,443,3.942473e+06,10,16,176,104.600000,447.312500,6.594845,0,1046
9,443,7.124851e+05,8,10,2454,557.875000,130.700000,25.263687,0,4463


In [8]:
top_features_loaded = pd.read_csv('top_features.csv')
print("Features your 3-class model expects:")
print(top_features_loaded['0'].tolist())

Features your 3-class model expects:
[' Destination Port', ' Bwd Header Length', ' Total Fwd Packets', ' Fwd Packet Length Mean', 'Total Length of Fwd Packets', ' Init_Win_bytes_backward', ' Fwd Packet Length Max', 'Init_Win_bytes_forward', ' Fwd Header Length.1', ' Max Packet Length', 'Fwd PSH Flags', ' Bwd Packet Length Mean', ' Flow Packets/s', ' SYN Flag Count', ' Subflow Fwd Bytes', ' Bwd Packet Length Std', ' Subflow Bwd Packets', ' Bwd Packets/s', ' Idle Min', ' Flow IAT Std']


In [9]:
import numpy as np

def calculate_flow_features_complete(flows):
    flow_features = []
    
    for flow_key, packets in flows.items():
        if len(packets) < 2:
            continue
        
        packets_sorted = sorted(packets, key=lambda x: x['time'])
        
        duration = (packets_sorted[-1]['time'] - packets_sorted[0]['time']) * 1_000_000
        duration = duration if duration > 0 else 1
        
        forward_src = packets_sorted[0]['src']
        fwd_packets = [p for p in packets_sorted if p['src'] == forward_src]
        bwd_packets = [p for p in packets_sorted if p['src'] != forward_src]
        
        fwd_lengths = [p['length'] for p in fwd_packets] or [0]
        bwd_lengths = [p['length'] for p in bwd_packets] or [0]
        all_lengths = [p['length'] for p in packets_sorted]
        
        # Inter-arrival times (time between consecutive packets)
        times = [p['time'] for p in packets_sorted]
        iat = [times[i+1]-times[i] for i in range(len(times)-1)] or [0]
        
        syn_count = sum(1 for p in packets_sorted if 'S' in p['flags'])
        psh_count_fwd = sum(1 for p in fwd_packets if 'P' in p['flags'])
        
        features = {
            'Destination_Port': packets_sorted[0]['dport'],
            'Bwd_Header_Length': len(bwd_packets) * 20,  # approx 20 bytes/header
            'Total_Fwd_Packets': len(fwd_packets),
            'Fwd_Packet_Length_Mean': np.mean(fwd_lengths),
            'Total_Length_Fwd_Packets': sum(fwd_lengths),
            'Init_Win_bytes_backward': bwd_lengths[0] if bwd_lengths else 0,  # approximation
            'Fwd_Packet_Length_Max': max(fwd_lengths),
            'Init_Win_bytes_forward': fwd_lengths[0] if fwd_lengths else 0,  # approximation
            'Fwd_Header_Length': len(fwd_packets) * 20,  # approx
            'Max_Packet_Length': max(all_lengths),
            'Fwd_PSH_Flags': psh_count_fwd,
            'Bwd_Packet_Length_Mean': np.mean(bwd_lengths),
            'Flow_Packets_s': len(packets_sorted) / (duration/1_000_000),
            'SYN_Flag_Count': syn_count,
            'Subflow_Fwd_Bytes': sum(fwd_lengths),  # approximation
            'Bwd_Packet_Length_Std': np.std(bwd_lengths),
            'Subflow_Bwd_Packets': len(bwd_packets),  # approximation
            'Bwd_Packets_s': len(bwd_packets) / (duration/1_000_000),
            'Idle_Min': min(iat) * 1_000_000 if iat else 0,  # approximation
            'Flow_IAT_Std': np.std(iat) * 1_000_000 if len(iat) > 1 else 0,
        }
        
        flow_features.append(features)
    
    return pd.DataFrame(flow_features)

df_complete = calculate_flow_features_complete(flows)
df_complete

,Destination_Port,Bwd_Header_Length,Total_Fwd_Packets,Fwd_Packet_Length_Mean,Total_Length_Fwd_Packets,Init_Win_bytes_backward,Fwd_Packet_Length_Max,Init_Win_bytes_forward,Fwd_Header_Length,Max_Packet_Length,Fwd_PSH_Flags,Bwd_Packet_Length_Mean,Flow_Packets_s,SYN_Flag_Count,Subflow_Fwd_Bytes,Bwd_Packet_Length_Std,Subflow_Bwd_Packets,Bwd_Packets_s,Idle_Min,Flow_IAT_Std
0,62767,20,1,54.000000,54,54,54,54,20,54,0,54.000000,20815.404467,0,54,0.000000,1,10407.702233,96.082687,0.000000
1,54008,160,37,509.000000,18833,54,1414,982,740,1414,35,165.750000,25.453582,0,18833,237.135378,8,4.525081,0.000000,114576.283701
2,443,180,9,623.111111,5608,108,1540,293,180,1540,6,129.000000,3.591958,0,5608,71.372264,9,1.795979,9.059906,567545.338176
3,443,220,12,915.916667,10991,54,3931,367,240,3931,8,240.181818,4.044802,0,10991,206.578102,11,1.934471,8.821487,378087.756223
4,443,620,14,295.357143,4135,66,1517,66,280,1517,6,425.096774,13.551768,2,4135,363.687242,31,9.335662,0.000000,153776.213580
5,443,240,9,1093.666667,9843,66,4317,66,180,4317,5,104.750000,6.341621,2,9843,67.798875,12,3.623784,0.953674,211237.555688
6,443,280,16,488.375000,7814,66,1927,66,320,6854,10,711.642857,16.574449,2,7814,1737.610248,14,7.734743,7.867813,88686.549158
7,443,140,7,195.857143,1371,66,571,66,140,4134,3,942.000000,27.710089,2,1371,1350.990748,7,13.855045,0.953674,50730.072780
8,443,320,10,104.600000,1046,124,176,176,200,1119,5,447.312500,6.594845,0,1046,397.001215,16,4.058366,0.953674,308643.436188
9,443,200,8,557.875000,4463,54,2454,121,160,2454,7,130.700000,25.263687,0,4463,144.112491,10,14.035382,0.953674,86119.401375


In [10]:
import joblib

# Load the 3-class model and scaler
model = joblib.load('threat_detection_model_3class.pkl')
scaler = joblib.load('scaler_3class.pkl')

print("Model and scaler loaded!")
print("Model expects", model.n_features_in_, "features")

Model and scaler loaded!
Model expects 20 features


In [11]:
# Get the exact column order the model was trained on
expected_columns = top_features_loaded['0'].tolist()

# Rename our columns to match exactly (mapping our names to CICIDS names)
column_mapping = {
    'Destination_Port': ' Destination Port',
    'Bwd_Header_Length': ' Bwd Header Length',
    'Total_Fwd_Packets': ' Total Fwd Packets',
    'Fwd_Packet_Length_Mean': ' Fwd Packet Length Mean',
    'Total_Length_Fwd_Packets': 'Total Length of Fwd Packets',
    'Init_Win_bytes_backward': ' Init_Win_bytes_backward',
    'Fwd_Packet_Length_Max': ' Fwd Packet Length Max',
    'Init_Win_bytes_forward': 'Init_Win_bytes_forward',
    'Fwd_Header_Length': ' Fwd Header Length.1',
    'Max_Packet_Length': ' Max Packet Length',
    'Fwd_PSH_Flags': 'Fwd PSH Flags',
    'Bwd_Packet_Length_Mean': ' Bwd Packet Length Mean',
    'Flow_Packets_s': ' Flow Packets/s',
    'SYN_Flag_Count': ' SYN Flag Count',
    'Subflow_Fwd_Bytes': ' Subflow Fwd Bytes',
    'Bwd_Packet_Length_Std': ' Bwd Packet Length Std',
    'Subflow_Bwd_Packets': ' Subflow Bwd Packets',
    'Bwd_Packets_s': ' Bwd Packets/s',
    'Idle_Min': ' Idle Min',
    'Flow_IAT_Std': ' Flow IAT Std'
}

df_complete_renamed = df_complete.rename(columns=column_mapping)

# Reorder columns to match exactly
df_final = df_complete_renamed[expected_columns]

print("Columns aligned correctly!")
df_final.head()

Columns aligned correctly!


,Destination Port,Bwd Header Length,Total Fwd Packets,Fwd Packet Length Mean,Total Length of Fwd Packets,Init_Win_bytes_backward,Fwd Packet Length Max,Init_Win_bytes_forward,Fwd Header Length.1,Max Packet Length,Fwd PSH Flags,Bwd Packet Length Mean,Flow Packets/s,SYN Flag Count,Subflow Fwd Bytes,Bwd Packet Length Std,Subflow Bwd Packets,Bwd Packets/s,Idle Min,Flow IAT Std
0,62767,20,1,54.000000,54,54,54,54,20,54,0,54.000000,20815.404467,0,54,0.000000,1,10407.702233,96.082687,0.000000
1,54008,160,37,509.000000,18833,54,1414,982,740,1414,35,165.750000,25.453582,0,18833,237.135378,8,4.525081,0.000000,114576.283701
2,443,180,9,623.111111,5608,108,1540,293,180,1540,6,129.000000,3.591958,0,5608,71.372264,9,1.795979,9.059906,567545.338176
3,443,220,12,915.916667,10991,54,3931,367,240,3931,8,240.181818,4.044802,0,10991,206.578102,11,1.934471,8.821487,378087.756223
4,443,620,14,295.357143,4135,66,1517,66,280,1517,6,425.096774,13.551768,2,4135,363.687242,31,9.335662,0.000000,153776.213580


In [12]:
# Scale the features
X_live_scaled = scaler.transform(df_final)

# Predict!
predictions = model.predict(X_live_scaled)
prediction_proba = model.predict_proba(X_live_scaled)

# Map predictions back to labels
label_map = {0: 'BENIGN', 1: 'FTP-Patator', 2: 'SSH-Patator'}
df_final['Prediction'] = [label_map[p] for p in predictions]
df_final['Confidence'] = [max(proba)*100 for proba in prediction_proba]

print("PREDICTIONS ON LIVE TRAFFIC:")
print(df_final[['Destination Port', 'Prediction', 'Confidence']])

PREDICTIONS ON LIVE TRAFFIC:


KeyError: "['Destination Port'] not in index"

In [13]:
print(df_final.columns.tolist())

[' Destination Port', ' Bwd Header Length', ' Total Fwd Packets', ' Fwd Packet Length Mean', 'Total Length of Fwd Packets', ' Init_Win_bytes_backward', ' Fwd Packet Length Max', 'Init_Win_bytes_forward', ' Fwd Header Length.1', ' Max Packet Length', 'Fwd PSH Flags', ' Bwd Packet Length Mean', ' Flow Packets/s', ' SYN Flag Count', ' Subflow Fwd Bytes', ' Bwd Packet Length Std', ' Subflow Bwd Packets', ' Bwd Packets/s', ' Idle Min', ' Flow IAT Std', 'Prediction', 'Confidence']


In [14]:
print("PREDICTIONS ON LIVE TRAFFIC:")
print(df_final[[' Destination Port', 'Prediction', 'Confidence']])

PREDICTIONS ON LIVE TRAFFIC:
    Destination Port Prediction  Confidence
0              62767     BENIGN       100.0
1              54008     BENIGN       100.0
2                443     BENIGN       100.0
3                443     BENIGN       100.0
4                443     BENIGN       100.0
5                443     BENIGN       100.0
6                443     BENIGN       100.0
7                443     BENIGN       100.0
8                443     BENIGN       100.0
9                443     BENIGN       100.0


In [15]:
def generate_alerts(df_predictions):
    print("="*50)
    print("THREAT MONITORING REPORT")
    print("="*50)
    
    for idx, row in df_predictions.iterrows():
        if row['Prediction'] == 'BENIGN':
            print(f"✅ Flow {idx}: Normal traffic on port {row[' Destination Port']} (Confidence: {row['Confidence']:.1f}%)")
        else:
            print(f"🚨 ALERT! Flow {idx}: {row['Prediction']} detected on port {row[' Destination Port']}! (Confidence: {row['Confidence']:.1f}%)")
    
    attack_count = (df_predictions['Prediction'] != 'BENIGN').sum()
    print("\n" + "="*50)
    print(f"Summary: {attack_count} potential threat(s) out of {len(df_predictions)} flows analyzed")
    print("="*50)

generate_alerts(df_final)

THREAT MONITORING REPORT
✅ Flow 0: Normal traffic on port 62767 (Confidence: 100.0%)
✅ Flow 1: Normal traffic on port 54008 (Confidence: 100.0%)
✅ Flow 2: Normal traffic on port 443 (Confidence: 100.0%)
✅ Flow 3: Normal traffic on port 443 (Confidence: 100.0%)
✅ Flow 4: Normal traffic on port 443 (Confidence: 100.0%)
✅ Flow 5: Normal traffic on port 443 (Confidence: 100.0%)
✅ Flow 6: Normal traffic on port 443 (Confidence: 100.0%)
✅ Flow 7: Normal traffic on port 443 (Confidence: 100.0%)
✅ Flow 8: Normal traffic on port 443 (Confidence: 100.0%)
✅ Flow 9: Normal traffic on port 443 (Confidence: 100.0%)

Summary: 0 potential threat(s) out of 10 flows analyzed


In [16]:
# Create a simulated SSH brute force flow based on Week 3 patterns
simulated_attack = pd.DataFrame([{
    ' Destination Port': 22,  # SSH port
    ' Bwd Header Length': 200,
    ' Total Fwd Packets': 11,
    ' Fwd Packet Length Mean': 45.0,
    'Total Length of Fwd Packets': 500,
    ' Init_Win_bytes_backward': 64,
    ' Fwd Packet Length Max': 60,
    'Init_Win_bytes_forward': 64,
    ' Fwd Header Length.1': 220,
    ' Max Packet Length': 70,
    'Fwd PSH Flags': 0,
    ' Bwd Packet Length Mean': 40.0,
    ' Flow Packets/s': 15435.0,  # matches SSH-Patator average from Week 3!
    ' SYN Flag Count': 0,  # matches SSH-Patator pattern from Week 3!
    ' Subflow Fwd Bytes': 500,
    ' Bwd Packet Length Std': 10.0,
    ' Subflow Bwd Packets': 11,
    ' Bwd Packets/s': 5000.0,
    ' Idle Min': 0,
    ' Flow IAT Std': 100.0
}])

# Predict on this simulated attack
X_sim_scaled = scaler.transform(simulated_attack)
sim_prediction = model.predict(X_sim_scaled)
sim_proba = model.predict_proba(X_sim_scaled)

print("Simulated SSH attack prediction:", label_map[sim_prediction[0]])
print("Confidence:", max(sim_proba[0])*100, "%")

Simulated SSH attack prediction: BENIGN
Confidence: 96.0 %


In [17]:
# Let's check real SSH-Patator examples from our actual dataset
df_check = pd.read_csv('combined_dataset.csv')
ssh_examples = df_check[df_check[' Label'] == 'SSH-Patator'][top_features_loaded['0'].tolist()]
print(ssh_examples.head(3))

         Destination Port   Bwd Header Length   Total Fwd Packets  \
691907                 22                   0                   2   
691908                 22                 584                  15   
691969                 22                 616                  18   

         Fwd Packet Length Mean  Total Length of Fwd Packets  \
691907                 0.000000                            0   
691908                99.733333                         1496   
691969                82.222222                         1480   

         Init_Win_bytes_backward   Fwd Packet Length Max  \
691907                        -1                       0   
691908                       257                     640   
691969                       247                     640   

        Init_Win_bytes_forward   Fwd Header Length.1   Max Packet Length  \
691907                     259                    64                   0   
691908                   29200                   488                 976 

In [18]:
# Use the actual second example (looks like a complete attack flow)
real_ssh_attack = ssh_examples.iloc[[1]]  # row 691908

print("Real SSH-Patator example:")
print(real_ssh_attack)

X_real_scaled = scaler.transform(real_ssh_attack)
real_prediction = model.predict(X_real_scaled)
real_proba = model.predict_proba(X_real_scaled)

print("\nPrediction:", label_map[real_prediction[0]])
print("Confidence:", max(real_proba[0])*100, "%")


Real SSH-Patator example:
         Destination Port   Bwd Header Length   Total Fwd Packets  \
691908                 22                 584                  15   

         Fwd Packet Length Mean  Total Length of Fwd Packets  \
691908                99.733333                         1496   

         Init_Win_bytes_backward   Fwd Packet Length Max  \
691908                       257                     640   

        Init_Win_bytes_forward   Fwd Header Length.1   Max Packet Length  \
691908                   29200                   488                 976   

        Fwd PSH Flags   Bwd Packet Length Mean   Flow Packets/s  \
691908              0               127.611111         5.680986   

         SYN Flag Count   Subflow Fwd Bytes   Bwd Packet Length Std  \
691908                0                1496              288.171342   

         Subflow Bwd Packets   Bwd Packets/s   Idle Min   Flow IAT Std  
691908                    18         3.09872          0    534015.4383  

Predict

In [19]:
import datetime

def real_time_monitor(duration=10, log_file='threat_log.csv'):
    print(f"🔍 Starting real-time monitoring for {duration} seconds...")
    
    flows = capture_and_extract_features(duration=duration)
    df_feat = calculate_flow_features_complete(flows)
    
    if len(df_feat) == 0:
        print("No flows captured.")
        return
    
    df_feat_renamed = df_feat.rename(columns=column_mapping)
    df_feat_final = df_feat_renamed[expected_columns]
    
    X_scaled = scaler.transform(df_feat_final)
    preds = model.predict(X_scaled)
    probas = model.predict_proba(X_scaled)
    
    df_feat_final['Prediction'] = [label_map[p] for p in preds]
    df_feat_final['Confidence'] = [max(p)*100 for p in probas]
    df_feat_final['Timestamp'] = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    
    generate_alerts(df_feat_final)
    
    # Log to file (append mode)
    import os
    file_exists = os.path.isfile(log_file)
    df_feat_final.to_csv(log_file, mode='a', header=not file_exists, index=False)
    print(f"\n📝 Results logged to {log_file}")
    
    return df_feat_final

# Run it!
results = real_time_monitor(duration=10)


🔍 Starting real-time monitoring for 10 seconds...
Capturing traffic for 10 seconds...
Captured 2944 packets across 44 flows
THREAT MONITORING REPORT
✅ Flow 0: Normal traffic on port 443 (Confidence: 100.0%)
✅ Flow 1: Normal traffic on port 443 (Confidence: 100.0%)
✅ Flow 2: Normal traffic on port 443 (Confidence: 100.0%)
✅ Flow 3: Normal traffic on port 443 (Confidence: 100.0%)
✅ Flow 4: Normal traffic on port 53989 (Confidence: 100.0%)
✅ Flow 5: Normal traffic on port 57226 (Confidence: 100.0%)
✅ Flow 6: Normal traffic on port 443 (Confidence: 100.0%)
✅ Flow 7: Normal traffic on port 443 (Confidence: 100.0%)
✅ Flow 8: Normal traffic on port 443 (Confidence: 100.0%)
✅ Flow 9: Normal traffic on port 443 (Confidence: 100.0%)
✅ Flow 10: Normal traffic on port 443 (Confidence: 100.0%)
✅ Flow 11: Normal traffic on port 443 (Confidence: 100.0%)
✅ Flow 12: Normal traffic on port 443 (Confidence: 100.0%)
✅ Flow 13: Normal traffic on port 443 (Confidence: 100.0%)
✅ Flow 14: Normal traffic on po

In [20]:
log_check = pd.read_csv('threat_log.csv')
print("Total logged entries:", len(log_check))
log_check.tail()

Total logged entries: 40


,Destination Port,Bwd Header Length,Total Fwd Packets,Fwd Packet Length Mean,Total Length of Fwd Packets,Init_Win_bytes_backward,Fwd Packet Length Max,Init_Win_bytes_forward,Fwd Header Length.1,Max Packet Length,...,SYN Flag Count,Subflow Fwd Bytes,Bwd Packet Length Std,Subflow Bwd Packets,Bwd Packets/s,Idle Min,Flow IAT Std,Prediction,Confidence,Timestamp
35,443,60,1,66.000000,66,66,66,66,20,66,...,4,66,1.885618,3,2.321365,277092.933655,180152.263988,BENIGN,100.0,2026-06-29 11:59:14
36,443,120,7,470.857143,3296,66,2238,66,140,2774,...,2,3296,993.069386,6,4.717293,2.145767,219895.566590,BENIGN,100.0,2026-06-29 11:59:14
37,443,60,1,66.000000,66,66,66,66,20,66,...,4,66,1.885618,3,2.338210,197485.923767,263234.081014,BENIGN,100.0,2026-06-29 11:59:14
38,443,20,4,839.250000,3357,66,1823,66,80,1823,...,2,3357,0.000000,1,1.195670,334.024429,230490.325126,BENIGN,100.0,2026-06-29 11:59:14
39,443,20,4,830.000000,3320,66,1786,66,80,1786,...,2,3320,0.000000,1,1.350147,147.104263,205547.754310,BENIGN,100.0,2026-06-29 11:59:14


In [22]:
print("Files in this session:")
print("- threat_log.csv (your monitoring logs)")
print("- Week5_Realtime_Monitoring.ipynb")

Files in this session:
- threat_log.csv (your monitoring logs)
- Week5_Realtime_Monitoring.ipynb
